In [ ]:
part0 = pd.read_csv('fraudTrain_part0.csv')
part1 = pd.read_csv('fraudTrain_part1.csv')
part2 = pd.read_csv('fraudTrain_part2.csv')
df = pd.concat([part0,part1,part2],axis=0)

In [ ]:
df2 = feature_eng(df)
fraud_data_df = df2[cols]
fraud_df = one_hot_encoded_cat(fraud_data_df)

In [ ]:
non_fraud_class = fraud_df[fraud_df['is_fraud'] == 0]
fraud_class = fraud_df[fraud_df['is_fraud'] == 1]
non_fraud_count,fraud_count=fraud_df['is_fraud'].value_counts()
print('The number of observations in non_fraud_class:', non_fraud_count)
print('The number of observations in fraud_class:', fraud_count)

In [ ]:
non_fraud_under = non_fraud_class.sample(fraud_count)
under_sampled = pd.concat([non_fraud_under, fraud_class], axis=0)
X_under = under_sampled.drop('is_fraud',axis=1)
y_under = under_sampled['is_fraud']

In [ ]:
X_train_under, X_test_under, y_train_under, y_test_under = train_test_split(X_under, y_under, random_state=0,train_size=0.8)

In [ ]:
param_log = {'C': np.logspace(-4, 4, 4), 'penalty': ['l1', 'l2']}
log_grid = GridSearchCV(LogisticRegression(),param_grid=param_log, n_jobs=-1)
log_grid.fit(X_train_under, y_train_under)
prediction_log = log_grid.predict(X_test_under)

In [ ]:
conf_mat_log = reporter(y_test_under,prediction_log)

In [ ]:
param_dt = {'max_depth': [3, 5, 10],'min_samples_split': [2, 4, 6],'criterion': ['gini', 'entropy']}
dt_grid = GridSearchCV(DecisionTreeClassifier(),param_grid=param_dt, n_jobs=-1)
dt_grid.fit(X_train_under, y_train_under)
prediction_dt = dt_grid.predict(X_test_under)

In [ ]:
conf_mat_dt = reporter(y_test_under,prediction_dt)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_rf = {'n_estimators':[20,50,100] ,'max_depth':[3,5,10],'min_samples_split':[2,4,6],'max_features':['auto', 'sqrt', 'log2']}
rf_grid = GridSearchCV(RandomForestClassifier(),param_grid=param_rf, n_jobs=-1)
rf_grid.fit(X_train_under, y_train_under)
prediction_rf = rf_grid.predict(X_test_under)

In [ ]:
conf_mat_rf = reporter(y_test_under,prediction_rf)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

param_boost = {'learning_rate': [0.01, 0.1],'max_depth': [3, 5, 7],'subsample': [0.5, 0.7],'colsample_bytree': [0.5, 0.7],'n_estimators': [10, 20, 30]}
boost_grid = RandomizedSearchCV(XGBClassifier(),param_boost, n_jobs=-1)
boost_grid.fit(X_train_under, y_train_under)
prediction_boost = boost_grid.predict(X_test_under)

In [ ]:
conf_mat_boost = reporter(y_test_under,prediction_boost)